# API Load Test

Manual load test against the **API Gateway** (stage `v1`), in **two comparable sessions**:

| Session | Users | Origin |
|--------|----------|--------|
| **1 — Known** | With history in `events.csv` | `u_0000` … `u_0499` |
| **2 — Cold start** | Absent from CSV | `u_0500` … (S3 fallback) |

Each session runs **4 rounds** with **5 concurrent requests** (`GET /recommendations/{user_id}`).
At the end we compare client-side latencies and Prometheus metrics from `/metrics`.

Configure (optional):

```bash
export RECOMMENDATIONS_API_BASE_URL="https://<api-id>.execute-api.us-east-1.amazonaws.com/v1"
export RECOMMENDATIONS_API_KEY="<your-api-key>"
export LOAD_TEST_BATCH_SIZE=5
export LOAD_TEST_ROUNDS=4
```

In [13]:
import json
import os
import statistics
import subprocess
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass, field
from pathlib import Path

import httpx
import pandas as pd

PROJECT_ROOT = Path("..").resolve()
TERRAFORM_DIR = PROJECT_ROOT / "terraform"
EVENTS_CSV = PROJECT_ROOT / "data" / "events.csv"
AWS_REGION = os.getenv("AWS_REGION", "us-east-1")
API_STAGE = "v1"
BATCH_SIZE = int(os.getenv("LOAD_TEST_BATCH_SIZE", "5"))
ROUNDS = int(os.getenv("LOAD_TEST_ROUNDS", "4"))
REQUESTS_PER_SESSION = BATCH_SIZE * ROUNDS


def _terraform_output(name: str) -> str | None:
    try:
        return subprocess.check_output(
            ["terraform", f"-chdir={TERRAFORM_DIR}", "output", "-raw", name],
            text=True,
            stderr=subprocess.DEVNULL,
        ).strip()
    except (subprocess.CalledProcessError, FileNotFoundError):
        return None


def normalize_api_base_url(base_url: str) -> str:
    normalized = base_url.rstrip("/")
    if normalized.endswith(f"/{API_STAGE}"):
        return normalized
    if "execute-api" in normalized:
        return f"{normalized}/{API_STAGE}"
    return normalized


def load_api_config() -> tuple[str, str]:
    base_url = os.getenv("RECOMMENDATIONS_API_BASE_URL")
    api_key = os.getenv("RECOMMENDATIONS_API_KEY")

    if not base_url:
        base_url = _terraform_output("recommendations_api_gateway_endpoint")

    if not api_key:
        api_key = _terraform_output("recommendations_api_key")

    if not api_key:
        param_name = _terraform_output("recommendations_api_key_ssm_parameter")
        if param_name:
            import boto3

            ssm = boto3.client("ssm", region_name=AWS_REGION)
            api_key = ssm.get_parameter(Name=param_name, WithDecryption=True)[
                "Parameter"
            ]["Value"]

    if not base_url or not api_key:
        raise RuntimeError(
            "Defina RECOMMENDATIONS_API_BASE_URL e RECOMMENDATIONS_API_KEY "
            "ou aplique o Terraform e configure credenciais AWS."
        )

    return normalize_api_base_url(base_url), api_key


def load_user_pools(events_path: Path) -> tuple[list[str], list[str]]:
    """Return (known_users, cold_start_users) derived from events.csv."""
    events = pd.read_csv(events_path)
    known_users = sorted(events["user_id"].astype(str).unique().tolist())
    known_set = set(known_users)

    if len(known_users) < REQUESTS_PER_SESSION:
        raise ValueError(
            f"events.csv has {len(known_users)} users but "
            f"{REQUESTS_PER_SESSION} unique ids are required per session."
        )

    cold_start_users: list[str] = []
    candidate_index = 500
    while len(cold_start_users) < REQUESTS_PER_SESSION:
        candidate = f"u_{candidate_index:04d}"
        if candidate not in known_set:
            cold_start_users.append(candidate)
        candidate_index += 1

    return known_users, cold_start_users


def parse_prometheus_metrics(text: str) -> dict[str, float]:
    metrics: dict[str, float] = {}
    for raw_line in text.splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#"):
            continue
        if "{" in line:
            value = float(line.rsplit(" ", 1)[1])
            if 'quantile="0.5"' in line:
                metrics["recommendations_api_latency_ms_p50"] = value
            elif 'quantile="0.95"' in line:
                metrics["recommendations_api_latency_ms_p95"] = value
            continue
        name, value = line.rsplit(" ", 1)
        metrics[name] = float(value)
    return metrics


API_BASE_URL, API_KEY = load_api_config()
HEADERS = {"Accept": "application/json", "x-api-key": API_KEY}
KNOWN_USERS, COLD_START_USERS = load_user_pools(EVENTS_CSV)

print(f"API base URL:     {API_BASE_URL}")
print(f"Batch size:       {BATCH_SIZE} requests simultâneas")
print(f"Rounds/session:   {ROUNDS}")
print(f"Reqs/session:     {REQUESTS_PER_SESSION}")
print(f"Known users:      {len(KNOWN_USERS)} (ex.: {KNOWN_USERS[0]} … {KNOWN_USERS[-1]})")
print(f"Cold start users: {len(COLD_START_USERS)} (ex.: {COLD_START_USERS[0]} … {COLD_START_USERS[-1]})")

API base URL:     https://swhf0h3h2a.execute-api.us-east-1.amazonaws.com/v1
Batch size:       5 requests simultâneas
Rounds/session:   4
Reqs/session:     20
Known users:      500 (ex.: u_0000 … u_0499)
Cold start users: 20 (ex.: u_0500 … u_0519)


In [14]:
@dataclass(frozen=True)
class RequestResult:
    session: str
    round_num: int
    user_id: str
    status_code: int
    latency_ms: float
    cold_start_flag: bool | None = None
    error: str = ""


@dataclass
class SessionReport:
    name: str
    user_pool_label: str
    results: list[RequestResult] = field(default_factory=list)
    metrics_before: dict[str, float] = field(default_factory=dict)
    metrics_after: dict[str, float] = field(default_factory=dict)

    def latencies_ms(self) -> list[float]:
        return [item.latency_ms for item in self.results if item.status_code == 200]

    def summary(self) -> dict[str, float | int | str]:
        latencies = self.latencies_ms()
        ok = sum(1 for item in self.results if item.status_code == 200)
        errors = len(self.results) - ok
        cold_flags = [item.cold_start_flag for item in self.results if item.cold_start_flag is not None]
        payload: dict[str, float | int | str] = {
            "session": self.name,
            "user_pool": self.user_pool_label,
            "requests": len(self.results),
            "ok": ok,
            "errors": errors,
        }
        if latencies:
            payload["client_p50_ms"] = round(statistics.median(latencies), 2)
            payload["client_p95_ms"] = round(
                statistics.quantiles(latencies, n=20)[18]
                if len(latencies) >= 2
                else latencies[0],
                2,
            )
            payload["client_avg_ms"] = round(statistics.mean(latencies), 2)
        if cold_flags:
            payload["cold_start_responses"] = sum(1 for flag in cold_flags if flag)
        if self.metrics_before and self.metrics_after:
            payload["server_requests_delta"] = int(
                self.metrics_after.get("recommendations_api_requests_total", 0)
                - self.metrics_before.get("recommendations_api_requests_total", 0)
            )
            payload["server_cold_start_delta"] = int(
                self.metrics_after.get("recommendations_api_cold_start_total", 0)
                - self.metrics_before.get("recommendations_api_cold_start_total", 0)
            )
            before_p50 = self.metrics_before.get("recommendations_api_latency_ms_p50")
            after_p50 = self.metrics_after.get("recommendations_api_latency_ms_p50")
            before_p95 = self.metrics_before.get("recommendations_api_latency_ms_p95")
            after_p95 = self.metrics_after.get("recommendations_api_latency_ms_p95")
            if before_p50 is not None and after_p50 is not None:
                payload["server_p50_ms_after"] = round(after_p50, 2)
            if before_p95 is not None and after_p95 is not None:
                payload["server_p95_ms_after"] = round(after_p95, 2)
        return payload


def users_for_round(user_pool: list[str], round_num: int) -> list[str]:
    """Pick BATCH_SIZE distinct users for one round, rotating through the pool."""
    start = ((round_num - 1) * BATCH_SIZE) % len(user_pool)
    return [user_pool[(start + slot) % len(user_pool)] for slot in range(BATCH_SIZE)]


def fire_request(
    client: httpx.Client,
    *,
    session: str,
    round_num: int,
    user_id: str,
) -> RequestResult:
    started = time.perf_counter()
    try:
        response = client.get(f"/recommendations/{user_id}", headers=HEADERS)
        latency_ms = (time.perf_counter() - started) * 1000
        cold_start_flag = None
        if response.status_code == 200:
            cold_start_flag = bool(response.json().get("cold_start_flag"))
        return RequestResult(
            session=session,
            round_num=round_num,
            user_id=user_id,
            status_code=response.status_code,
            latency_ms=latency_ms,
            cold_start_flag=cold_start_flag,
        )
    except httpx.HTTPError as error:
        latency_ms = (time.perf_counter() - started) * 1000
        return RequestResult(
            session=session,
            round_num=round_num,
            user_id=user_id,
            status_code=0,
            latency_ms=latency_ms,
            error=str(error),
        )


def fetch_metrics(client: httpx.Client) -> dict[str, float]:
    response = client.get("/metrics", headers={"Accept": "text/plain", **HEADERS})
    response.raise_for_status()
    return parse_prometheus_metrics(response.text)


def run_load_session(
    client: httpx.Client,
    *,
    session_name: str,
    user_pool: list[str],
    user_pool_label: str,
) -> SessionReport:
    report = SessionReport(name=session_name, user_pool_label=user_pool_label)
    report.metrics_before = fetch_metrics(client)

    for round_num in range(1, ROUNDS + 1):
        round_users = users_for_round(user_pool, round_num)
        round_started = time.perf_counter()
        round_results: list[RequestResult] = []

        with ThreadPoolExecutor(max_workers=BATCH_SIZE) as pool:
            futures = [
                pool.submit(
                    fire_request,
                    client,
                    session=session_name,
                    round_num=round_num,
                    user_id=user_id,
                )
                for user_id in round_users
            ]
            for future in as_completed(futures):
                round_results.append(future.result())

        report.results.extend(round_results)
        ok = sum(1 for item in round_results if item.status_code == 200)
        errors = len(round_results) - ok
        latencies = [item.latency_ms for item in round_results if item.status_code == 200]
        cold = sum(1 for item in round_results if item.cold_start_flag)
        round_elapsed = (time.perf_counter() - round_started) * 1000

        if latencies:
            p50 = statistics.median(latencies)
            p95 = (
                statistics.quantiles(latencies, n=20)[18]
                if len(latencies) >= 2
                else latencies[0]
            )
            print(
                f"[{session_name}] round {round_num}/{ROUNDS}: ok={ok} errors={errors} "
                f"cold_start={cold} p50={p50:.1f}ms p95={p95:.1f}ms wall={round_elapsed:.0f}ms"
            )
        else:
            print(
                f"[{session_name}] round {round_num}/{ROUNDS}: ok={ok} errors={errors} "
                f"wall={round_elapsed:.0f}ms"
            )

    report.metrics_after = fetch_metrics(client)
    return report

## Session 1 — known users (`events.csv`)

Users with real history in the dataset (`u_0000` … `u_0499`). Expect `cold_start_flag=false` and DynamoDB reads.

In [15]:
with httpx.Client(base_url=API_BASE_URL, timeout=60.0) as client:
    known_report = run_load_session(
        client,
        session_name="known_users",
        user_pool=KNOWN_USERS,
        user_pool_label="events.csv",
    )

print("\n--- Sessão 1 summary ---")
print(json.dumps(known_report.summary(), indent=2))

[known_users] round 1/4: ok=5 errors=0 cold_start=0 p50=476.3ms p95=513.6ms wall=515ms
[known_users] round 2/4: ok=5 errors=0 cold_start=0 p50=171.2ms p95=182.8ms wall=179ms
[known_users] round 3/4: ok=5 errors=0 cold_start=0 p50=188.4ms p95=197.1ms wall=195ms
[known_users] round 4/4: ok=5 errors=0 cold_start=0 p50=193.9ms p95=237.8ms wall=221ms

--- Sessão 1 summary ---
{
  "session": "known_users",
  "user_pool": "events.csv",
  "requests": 20,
  "ok": 20,
  "errors": 0,
  "client_p50_ms": 190.15,
  "client_p95_ms": 513.01,
  "client_avg_ms": 241.04,
  "cold_start_responses": 0,
  "server_requests_delta": 20,
  "server_cold_start_delta": 0,
  "server_p50_ms_after": 687.98,
  "server_p95_ms_after": 1226.93
}


## Session 2 — cold start

Users **absent** from `events.csv` (`u_0500+`). Expect `cold_start_flag=true` and fallback via S3 catalog.

In [16]:
with httpx.Client(base_url=API_BASE_URL, timeout=60.0) as client:
    cold_report = run_load_session(
        client,
        session_name="cold_start",
        user_pool=COLD_START_USERS,
        user_pool_label="not in events.csv",
    )

print("\n--- Sessão 2 summary ---")
print(json.dumps(cold_report.summary(), indent=2))

[cold_start] round 1/4: ok=5 errors=0 cold_start=5 p50=364.2ms p95=384.0ms wall=386ms
[cold_start] round 2/4: ok=5 errors=0 cold_start=5 p50=163.0ms p95=195.8ms wall=184ms
[cold_start] round 3/4: ok=5 errors=0 cold_start=5 p50=143.2ms p95=148.6ms wall=147ms
[cold_start] round 4/4: ok=5 errors=0 cold_start=5 p50=141.9ms p95=149.4ms wall=148ms

--- Sessão 2 summary ---
{
  "session": "cold_start",
  "user_pool": "not in events.csv",
  "requests": 20,
  "ok": 20,
  "errors": 0,
  "client_p50_ms": 144.95,
  "client_p95_ms": 383.74,
  "client_avg_ms": 191.55,
  "cold_start_responses": 20,
  "server_requests_delta": 20,
  "server_cold_start_delta": 20,
  "server_p50_ms_after": 670.32,
  "server_p95_ms_after": 1225.56
}


## Comparison and Prometheus metrics

Compare **client-side** latencies per session and display the final snapshot from `/metrics` (accumulated counters in the container).

In [17]:
comparison = {
    "known_users": known_report.summary(),
    "cold_start": cold_report.summary(),
    "delta_client_p50_ms": round(
        float(cold_report.summary().get("client_p50_ms", 0))
        - float(known_report.summary().get("client_p50_ms", 0)),
        2,
    ),
    "delta_client_p95_ms": round(
        float(cold_report.summary().get("client_p95_ms", 0))
        - float(known_report.summary().get("client_p95_ms", 0)),
        2,
    ),
}

print("--- Comparação entre sessões ---")
print(json.dumps(comparison, indent=2))

with httpx.Client(base_url=API_BASE_URL, timeout=60.0) as client:
    metrics_response = client.get("/metrics", headers={"Accept": "text/plain", **HEADERS})

print(f"\nStatus: {metrics_response.status_code}")
print(f"Content-Type: {metrics_response.headers.get('content-type')}")
print("\n--- /metrics (Prometheus) ---")
print(metrics_response.text)

final_metrics = parse_prometheus_metrics(metrics_response.text)
print("\n--- parsed summary ---")
print(
    json.dumps(
        {
            "requests_total": final_metrics.get("recommendations_api_requests_total"),
            "errors_total": final_metrics.get("recommendations_api_errors_total"),
            "cold_start_total": final_metrics.get("recommendations_api_cold_start_total"),
            "latency_avg_ms": final_metrics.get("recommendations_api_latency_avg_ms"),
            "latency_p50_ms": final_metrics.get("recommendations_api_latency_ms_p50"),
            "latency_p95_ms": final_metrics.get("recommendations_api_latency_ms_p95"),
        },
        indent=2,
    )
)

assert metrics_response.status_code == 200
assert known_report.summary()["cold_start_responses"] == 0
assert cold_report.summary()["cold_start_responses"] == cold_report.summary()["ok"]
print("\nLoad test comparison passed.")

--- Comparação entre sessões ---
{
  "known_users": {
    "session": "known_users",
    "user_pool": "events.csv",
    "requests": 20,
    "ok": 20,
    "errors": 0,
    "client_p50_ms": 190.15,
    "client_p95_ms": 513.01,
    "client_avg_ms": 241.04,
    "cold_start_responses": 0,
    "server_requests_delta": 20,
    "server_cold_start_delta": 0,
    "server_p50_ms_after": 687.98,
    "server_p95_ms_after": 1226.93
  },
  "cold_start": {
    "session": "cold_start",
    "user_pool": "not in events.csv",
    "requests": 20,
    "ok": 20,
    "errors": 0,
    "client_p50_ms": 144.95,
    "client_p95_ms": 383.74,
    "client_avg_ms": 191.55,
    "cold_start_responses": 20,
    "server_requests_delta": 20,
    "server_cold_start_delta": 20,
    "server_p50_ms_after": 670.32,
    "server_p95_ms_after": 1225.56
  },
  "delta_client_p50_ms": -45.2,
  "delta_client_p95_ms": -129.27
}

Status: 200
Content-Type: text/plain; version=0.0.4; charset=utf-8

--- /metrics (Prometheus) ---
# HELP rec